# Modelo Elegido: Random Forest

In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

ruta = 'work/autotec/app/datos_autotec.csv'
df = pd.read_csv(ruta)

objetivo = 'precio'
df = df.dropna(subset=[objetivo]).copy()

X = df.drop(columns=[objetivo])
y = df[objetivo]

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

modelo = RandomForestRegressor(random_state=42, n_jobs=-1)

pipe = Pipeline([
    ('prep', preprocessor),
    ('model', modelo)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

param_grid = {
    'model__n_estimators': [200, 300],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1,
    error_score='raise'
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print('Mejores parámetros:', grid.best_params_)
print('MAE:', mae)
print('RMSE:', rmse)
print('R2:', r2)

resultados = pd.DataFrame({'y_real': y_test.values, 'y_predicho': y_pred})
resultados.to_csv('resultados_RandomForest.csv', index=False)

metricas = pd.DataFrame([{
    'modelo': 'RandomForestRegressor',
    'mae': mae,
    'rmse': rmse,
    'r2': r2,
    'mejores_parametros': str(grid.best_params_)
}])


Fitting 3 folds for each of 24 candidates, totalling 72 fits
Mejores parámetros: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 200}
MAE: 2768339.0854271357
RMSE: 4893661.213834106
R2: 0.7384935433441373


['modelo_precio_autotec.joblib']

## Resultado del Modelo

In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predicciones ya calculadas: y_pred
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

# "Accuracy" aproximada para regresión: porcentaje de predicciones dentro de un margen
margen = 0.10
accuracy_aprox = np.mean(np.abs(y_test - y_pred) / y_test <= margen) * 100

print("=" * 60)
print("EVALUACIÓN DEL MODELO DE REGRESIÓN")
print("=" * 60)
print(f"Coeficiente de determinación (R2): {r2:.4f}")
print(f"Error absoluto medio (MAE): {mae:,.2f}")
print(f"Raíz del error cuadrático medio (RMSE): {rmse:,.2f}")
print(f"Accuracy aproximado dentro de ±{int(margen*100)}%: {accuracy_aprox:.2f}%")
print("=" * 60)

EVALUACIÓN DEL MODELO DE REGRESIÓN
Coeficiente de determinación (R2): 0.7385
Error absoluto medio (MAE): 2,768,339.09
Raíz del error cuadrático medio (RMSE): 4,893,661.21
Accuracy aproximado dentro de ±10%: 41.46%


## Interpretación:

## Métricas obtenidas
Las métricas reportadas para el modelo fueron las siguientes:

| Métrica | Valor obtenido | Interpretación |
|---------|----------------|----------------|
| Coeficiente de determinación \(R^2\) | 0.7385 | El modelo explica aproximadamente el 73.85% de la variabilidad del precio del vehículo. |
| Error absoluto medio (MAE) | 2,768,339.09 | En promedio, la predicción del modelo se desvía en cerca de 2.77 millones respecto al precio real. |
| Raíz del error cuadrático medio (RMSE) | 4,893,661.21 | Existen algunos errores grandes que elevan el error global del modelo. |

El coeficiente de determinación \(R^2\) es una métrica estándar para evaluar el ajuste de modelos de regresión y valores más cercanos a 1 indican una mayor capacidad explicativa. El error absoluto medio representa la diferencia promedio, en valor absoluto, entre lo predicho y lo real, por lo que su lectura es directa y útil para estimar el error típico del modelo. El RMSE también mide error de predicción, pero penaliza más fuertemente los errores grandes, por lo que suele aumentar cuando existen observaciones difíciles o atípicas.

## Interpretación del desempeño
El modelo de regresión obtenido presenta un desempeño satisfactorio para la estimación del precio de vehículos usados. 

-El coeficiente de determinación \(R^2 = 0.7385\) indica que el modelo explica una proporción importante de la variabilidad del precio,

-El MAE de 2,768,339.09 muestra que, en promedio, las predicciones se desvían en cerca de 2.77 millones respecto al valor real. 

-Por su parte, el RMSE de 4,893,661.21 evidencia la existencia de algunos errores más elevados en ciertos casos, lo que sugiere que el modelo responde bien en términos generales, aunque todavía enfrenta dificultades ante observaciones extremas o menos frecuentes del mercado.

### En conjunto, los resultados permiten concluir que el modelo es útil como apoyo para procesos de tasación y análisis de valor comercial, ya que capta patrones relevantes asociados al kilometraje, la antigüedad y la marca del vehículo.
